In [2]:
import json
import random
import commentjson
import re

my_label = 7
templatePath = 'intentTemplate/date_template.jsonc'
entityPath = 'entities/date.jsonc'
authorPath = 'entities/authors.jsonc'
entityMain = 'genre'
entityAuthor = 'AUTHOR'


def choose_josa(word, josa_with_batchim, josa_without_batchim):
    if not word:
        return ""
    last_char = word[-1]
    if '가' <= last_char <= '힣':
        has_batchim = (ord(last_char) - ord('가')) % 28 > 0
        return josa_with_batchim if has_batchim else josa_without_batchim
    return josa_without_batchim

def process_korean_template(template, title, author=""):
    text = template.replace('{}', title)
    text = text.replace('[]', author)
    text = text.replace('{이}', choose_josa(title, '이', ''))
    text = text.replace('{은}', choose_josa(title, '은', '는'))
    text = text.replace('{를}', choose_josa(title, '을', '를'))
    text = text.replace('{과}', choose_josa(title, '과', '와'))
    text = text.replace('{가}', choose_josa(title, '이', '가'))
    text = text.replace('{으로}', choose_josa(title, '으로', '로'))
    return text

def generate_ner_data(templates, titles, authors, current_intent_label, main_entity_type, author_entity_type):
    output_data = []
    format_keywords = ["책", "도서", "자료"]
    exclude_words = ["도서관", "자료실", "자료집", "자료형"]

    for template in templates:
        title = random.choice(titles) if titles else ""
        author = random.choice(authors) if authors else ""
        formatted_text = process_korean_template(template, title, author)

        current_entities = []

        if title:
            title_start = formatted_text.find(title)
            if title_start != -1:
                title_end = title_start + len(title)
                current_entities.append({
                    "start": title_start,
                    "end": title_end,
                    "type": main_entity_type
                })

        if author:
            author_start = -1
            search_offset = 0
            while True: # 저자명과 제목명이 겹치지 않도록 기본적인 탐색
                temp_author_start = formatted_text.find(author, search_offset)
                if temp_author_start == -1:
                    break
                
                is_overlapping_with_title = False
                if title and title_start != -1:
                    title_range = range(title_start, title_start + len(title))
                    author_range = range(temp_author_start, temp_author_start + len(author))
                    if max(title_range.start, author_range.start) < min(title_range.stop, author_range.stop):
                        is_overlapping_with_title = True
                
                if not is_overlapping_with_title:
                    author_start = temp_author_start
                    break
                search_offset = temp_author_start + 1
            
            if author_start != -1:
                author_end = author_start + len(author)
                current_entities.append({
                    "start": author_start,
                    "end": author_end,
                    "type": author_entity_type
                })

        for keyword in format_keywords:
            start_pos = 0
            while True:
                keyword_start = formatted_text.find(keyword, start_pos)
                if keyword_start == -1:
                    break
                keyword_end = keyword_start + len(keyword)

                is_excluded = False
                for exclude_word in exclude_words:
                    # 키워드가 exclude_word의 일부인지 확인 (예: "도서" in "도서관")
                    # 좀 더 정확한 비교를 위해, exclude_word가 keyword_start 위치에서 시작하는지,
                    # 그리고 그 안에 keyword가 포함되는지 확인
                    if formatted_text.startswith(exclude_word, max(0, keyword_start - exclude_word.find(keyword) if keyword in exclude_word else 0)):
                         # keyword_start가 exclude_word 내의 keyword 위치와 일치하는지 확인
                        idx_in_exclude = exclude_word.find(keyword)
                        if idx_in_exclude != -1:
                            if keyword_start - idx_in_exclude >= 0 and \
                               formatted_text[keyword_start - idx_in_exclude : keyword_start - idx_in_exclude + len(exclude_word)] == exclude_word:
                                is_excluded = True
                                break
                if is_excluded:
                    start_pos = keyword_start + 1 
                    continue

                valid_endings = True
                if keyword_end < len(formatted_text):
                    next_char = formatted_text[keyword_end]
                    if '가' <= next_char <= '힣':
                        valid_suffixes = ['이', '가', '을', '를', '은', '는', '의', '에', '도', '만', '과', '와',
                                         '까지', '부터', '에서', '으로', '로', '들', '께', '랑', '이랑', '하고', '점']
                        found_valid_suffix = False
                        if not next_char.isalnum():
                            found_valid_suffix = True
                        else:
                            for suffix in valid_suffixes:
                                if formatted_text[keyword_end:].startswith(suffix):
                                    if len(formatted_text) == keyword_end + len(suffix) or \
                                       not formatted_text[keyword_end + len(suffix)].isalnum() or \
                                       not ('가' <= formatted_text[keyword_end + len(suffix)] <= '힣'):
                                        found_valid_suffix = True
                                        break
                        if not found_valid_suffix:
                            valid_endings = False
                
                valid_beginnings = True
                if keyword_start > 0:
                    prev_char = formatted_text[keyword_start-1]
                    if '가' <= prev_char <= '힣':
                        valid_beginnings = False

                if valid_endings and valid_beginnings:
                    is_overlapping_with_existing = False
                    new_entity_range = range(keyword_start, keyword_end)
                    for existing_entity in current_entities:
                        existing_entity_range = range(existing_entity["start"], existing_entity["end"])
                        if max(new_entity_range.start, existing_entity_range.start) < min(new_entity_range.stop, existing_entity_range.stop):
                            is_overlapping_with_existing = True
                            break
                    
                    if not is_overlapping_with_existing:
                        current_entities.append({
                            "start": keyword_start,
                            "end": keyword_end,
                            "type": "FORMAT"
                        })
                start_pos = keyword_end
        
        current_entities.sort(key=lambda x: x["start"])
        data_point = {"text": formatted_text, "intent": current_intent_label, "entities": current_entities}
        output_data.append(data_point)
    return output_data


try:
    with open(templatePath, 'r', encoding='utf-8') as f:
        my_templates = commentjson.load(f)
except FileNotFoundError:
    print(f"오류: 템플릿 파일({templatePath})을 찾을 수 없습니다.")
    my_templates = []

try:
    with open(entityPath, 'r', encoding='utf-8') as f:
        my_main_entities_from_file = commentjson.load(f) 
except FileNotFoundError:
    print(f"오류: 주요 엔티티 파일({entityPath})을 찾을 수 없습니다.")
    my_main_entities_from_file = []

try:
    with open(authorPath, 'r', encoding='utf-8') as f:
        my_authors = commentjson.load(f)
except FileNotFoundError:
    print(f"정보: 작가 파일({authorPath})을 찾을 수 없습니다. 작가 정보 없이 진행합니다.")
    my_authors = []



if my_templates and my_main_entities_from_file:
    generated_ner_data = generate_ner_data(
        my_templates,
        my_main_entities_from_file, 
        my_authors,
        my_label,       
        "DATE",         
        entityAuthor    
    )
else:
    generated_ner_data = []
    if not my_templates: print("템플릿 데이터가 없어 NER 데이터를 생성할 수 없습니다.")
    if not my_main_entities_from_file: print("주요 엔티티 데이터가 없어 NER 데이터를 생성할 수 없습니다.")

# NER 데이터 출력
ner_output = json.dumps(generated_ner_data, ensure_ascii=False, indent=2)
# 가장 바깥쪽의 [] 제거
ner_output_without_brackets = ner_output[1:-1].strip()
print(ner_output_without_brackets)


{
    "text": "야 8월 15일에 도서관 쉬냐?",
    "intent": 7,
    "entities": [
      {
        "start": 2,
        "end": 8,
        "type": "DATE"
      }
    ]
  },
  {
    "text": "2024년 12월 25일에 도서관 쉬는 날이야?",
    "intent": 7,
    "entities": [
      {
        "start": 0,
        "end": 13,
        "type": "DATE"
      }
    ]
  },
  {
    "text": "야 일주일 뒤 도서관 언제 쉬어?",
    "intent": 7,
    "entities": [
      {
        "start": 2,
        "end": 7,
        "type": "DATE"
      }
    ]
  },
  {
    "text": "야 도서관 휴관일이 언제냐?",
    "intent": 7,
    "entities": []
  },
  {
    "text": "도서관 언제쉼?",
    "intent": 7,
    "entities": []
  },
  {
    "text": "나 이번주 도서관 쌉가능?",
    "intent": 7,
    "entities": [
      {
        "start": 2,
        "end": 5,
        "type": "DATE"
      }
    ]
  },
  {
    "text": "오늘에 도서관 여냐?",
    "intent": 7,
    "entities": [
      {
        "start": 0,
        "end": 2,
        "type": "DATE"
      }
    ]
  },
  {
    "text": "야 2025년 1월 1일 도서관 문 열어?",
    "intent"